Modelo_B_correciones.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1mlggi47EYM4iYkfhmSCcrlLi0K2VE5uc

# Modelo B - SimCLR y Fine-tuning

## Instalación e importación de librerias

In [1]:
# =========================================================
# 1. Instalación y autenticación
# =========================================================
!pip -q install datasets huggingface_hub scikit-learn tensorboard

from huggingface_hub import notebook_login
notebook_login()


KeyboardInterrupt: 

In [ ]:
# =========================================================
# 2. Imports y configuración general
# =========================================================
import os
import math
import random
import json
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image
import io

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import torchvision
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from datasets import load_dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, average_precision_score
)
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

# -----------------------------
# Reproducibilidad
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# =========================================================
# 3. RUTAS
# =========================================================

PROJECT_ROOT = Path('/content/drive/MyDrive/TEC_SkinLesions_Project')

PROJECT_DIR = PROJECT_ROOT / 'model_b_outputs'
COMMON_SPLIT_DIR = PROJECT_ROOT / 'shared_data' / 'ISIC2019_MEL_NV_COMMON_SPLIT_SEED42'
COMMON_MANIFEST = COMMON_SPLIT_DIR / 'mel_nv_split_manifest.csv'

SIMCLR_CKPT_DIR = PROJECT_DIR / 'checkpoints_simclr'
FINETUNE_CKPT_DIR = PROJECT_DIR / 'checkpoints_finetune'
HISTORY_DIR = PROJECT_DIR / 'history'
FIGURES_DIR = PROJECT_DIR / 'figures'
EXPORT_DIR = PROJECT_DIR / 'exports_for_tcav'
LOG_DIR = PROJECT_DIR / 'tensorboard_logs'

for d in [PROJECT_DIR, COMMON_SPLIT_DIR, SIMCLR_CKPT_DIR, FINETUNE_CKPT_DIR, HISTORY_DIR, FIGURES_DIR, EXPORT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Model saved in:', PROJECT_DIR)


In [ ]:
# =========================================================
# 4. Hiperparámetros
# =========================================================
HF_DATASET_NAME = 'Atoany/ISIC2019'

# Imagen
IMG_SIZE = 224
# Antes: NUM_WORKERS = 0 (todo el decode/augment de imagenes corria en el
# proceso principal, bloqueando la GPU entre batches). Ahora se usan varios
# procesos worker para preparar batches en paralelo mientras la GPU entrena.
# Se deja 1 CPU libre para el proceso principal/Colab.
NUM_WORKERS = max(1, (os.cpu_count() or 2) - 1)
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None
print('NUM_WORKERS:', NUM_WORKERS)

# -----------------------------
# Preentrenamiento SimCLR
# -----------------------------
SIMCLR_EPOCHS = 200              # Recomendado: 100-200 para Colab; más si hay tiempo
BATCH_SIZE_SIMCLR = 128          # Prueba 128 o 256. Si falla por memoria, baja a 64.
GRAD_ACCUM_STEPS_SIMCLR = 1      # Simula batch efectivo: BATCH_SIZE_SIMCLR * GRAD_ACCUM_STEPS_SIMCLR
SIMCLR_LR = 3e-4
SIMCLR_WEIGHT_DECAY = 1e-4
SIMCLR_TEMPERATURE = 0.5
PROJECTION_DIM = 128
HIDDEN_DIM = 2048
WARMUP_EPOCHS_SIMCLR = 10

# -----------------------------
# Fine-tuning supervisado MEL vs NV
# -----------------------------
FINETUNE_EPOCHS = 50
BATCH_SIZE_FINETUNE = 64
GRAD_ACCUM_STEPS_FINETUNE = 1
FINETUNE_LR_BACKBONE = 1e-5
FINETUNE_LR_FC = 1e-4
FINETUNE_WEIGHT_DECAY = 5e-4
WARMUP_EPOCHS_FINETUNE = 3
EARLY_STOP_PATIENCE = 10
USE_CLASS_WEIGHTS = True

# -----------------------------
# Reanudación automática
# -----------------------------
AUTO_RESUME_SIMCLR = True
AUTO_RESUME_FINETUNE = True

# AMP
USE_AMP = True

# Guardar config
config = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (str, int, float, bool))}
with open(PROJECT_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

config


In [ ]:
# =========================================================
# 5. Cargar dataset desde Hugging Face
# =========================================================
dataset = load_dataset(HF_DATASET_NAME,
                       cache_dir="/content/drive/MyDrive/TEC/Research/SkinLesions/TCAV/RawDataset/")

# Guardamos el indice original de cada fila, en cada split, ANTES de
# cualquier filter/map. Esto nos permite, mas adelante, identificar
# exactamente que muestras se usan en SimCLR y excluir las que el
# manifiesto de Modelo A asigno a validation/test (evita fuga de datos).
# add_column preserva la columna a traves de futuros .filter()/.map().
for split in dataset.keys():
    dataset[split] = dataset[split].add_column('__orig_idx__', list(range(len(dataset[split]))))

print(dataset)
print('Splits:', dataset.keys())
print('Columnas train:', dataset['train'].column_names)
print('Ejemplo:', dataset['train'][0])


In [ ]:
# =========================================================
# 6. Detección robusta de columnas de imagen y etiqueta
# =========================================================
def detect_image_column(ds_split):
    sample = ds_split[0]
    for col, val in sample.items():
        if isinstance(val, Image.Image):
            return col
    # fallback común
    for col in ds_split.column_names:
        if col.lower() in ['image', 'img', 'isic_image']:
            return col
    raise ValueError('No se pudo detectar la columna de imagen. Ajusta IMAGE_COL manualmente.')


def detect_label_column(ds_split):
    candidates = ['label', 'labels', 'target', 'dx', 'diagnosis', 'class']
    for col in ds_split.column_names:
        if col.lower() in candidates:
            return col
    raise ValueError('No se pudo detectar la columna de etiqueta. Ajusta LABEL_COL manualmente.')

IMAGE_COL = detect_image_column(dataset['train'])
LABEL_COL = detect_label_column(dataset['train'])
print('IMAGE_COL:', IMAGE_COL)
print('LABEL_COL:', LABEL_COL)

# Obtener nombres de clases si el dataset los tiene como ClassLabel
features = dataset['train'].features
label_feature = features[LABEL_COL]
print('Label feature:', label_feature)

if hasattr(label_feature, 'names') and label_feature.names is not None:
    CLASS_NAMES = list(label_feature.names)
else:
    # Si las etiquetas son strings o ints sin ClassLabel
    vals = dataset['train'].unique(LABEL_COL)
    CLASS_NAMES = sorted([str(v) for v in vals])

print('CLASS_NAMES detectadas:', CLASS_NAMES)


In [ ]:
# =========================================================
# 7. Normalización de etiquetas ISIC2019
# =========================================================
# Este bloque intenta mapear las etiquetas a nombres estándar:
# MEL, NV, BCC, AK, BKL, DF, SCC, VASC.

CANONICAL = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'SCC', 'VASC']
ALIASES = {
    'MEL': ['MEL', 'mel', 'melanoma'],
    'NV': ['NV', 'nv', 'nevus', 'melanocytic nevus', 'melanocytic_nevus'],
    'BCC': ['BCC', 'bcc', 'basal cell carcinoma', 'basal_cell_carcinoma'],
    'AK': ['AK', 'ak', 'AKIEC', 'akiec', 'actinic keratosis', 'actinic_keratosis'],
    'BKL': ['BKL', 'bkl', 'benign keratosis', 'benign_keratosis'],
    'DF': ['DF', 'df', 'dermatofibroma'],
    'SCC': ['SCC', 'scc', 'squamous cell carcinoma', 'squamous_cell_carcinoma'],
    'VASC': ['VASC', 'vasc', 'vascular lesion', 'vascular_lesion'],
}

# Para datasets con ClassLabel numérico: índice -> nombre
id_to_raw_name = {}
if hasattr(dataset['train'].features[LABEL_COL], 'names') and dataset['train'].features[LABEL_COL].names is not None:
    for i, name in enumerate(dataset['train'].features[LABEL_COL].names):
        id_to_raw_name[i] = name


def canonical_label(example):
    raw = example[LABEL_COL]
    if isinstance(raw, (int, np.integer)) and raw in id_to_raw_name:
        raw_name = id_to_raw_name[int(raw)]
    else:
        raw_name = str(raw)
    raw_norm = raw_name.strip()
    for canon, aliases in ALIASES.items():
        if raw_norm in aliases or raw_norm.upper() == canon:
            example['dx_canon'] = canon
            return example
    # fallback: conservar mayúsculas
    example['dx_canon'] = raw_norm.upper()
    return example

# Agregar etiqueta canónica a todos los splits
for split in dataset.keys():
    dataset[split] = dataset[split].map(canonical_label)

print(dataset)
print('Clases canónicas train:', sorted(dataset['train'].unique('dx_canon')))


In [ ]:
# =========================================================
# 8. Preparar dataset para SimCLR (8 clases de ISIC2019)
# =========================================================
# SimCLR (preentrenamiento no supervisado) usa todas las imagenes de las
# 8 clases objetivo y no usa etiquetas, por lo que NO depende del split
# comun de Modelo A.
#
# IMPORTANTE: el split MEL vs NV para fine-tuning YA NO se calcula en este
# notebook. Se reconstruye en la celda siguiente a partir del manifiesto
# compartido que genera el Modelo A, para garantizar particiones train/
# validation/test IDENTICAS entre ambos modelos.

def filter_8_classes(ex):
    return ex['dx_canon'] in CANONICAL

simclr_dataset = DatasetDict({
    split: dataset[split].filter(filter_8_classes)
    for split in dataset.keys()
})

print('SimCLR dataset (8 clases, sin usar etiquetas):', simclr_dataset)


In [ ]:
# =========================================================
# 9. Cargar el split COMUN de Modelo A (obligatorio) para fine-tuning
# =========================================================
# Modelo B ya NO genera su propio split train/validation/test para
# MEL vs NV. En su lugar, lee el manifiesto compartido que crea el
# Modelo A (mismo pool MEL/NV, mismo seed=42, mismo split
# estratificado 70/15/15) y reconstruye, muestra por muestra, las
# particiones exactas usando (source_split, source_index).
#
# Si el manifiesto no existe todavia, este notebook se detiene con un
# error: primero debes correr el Modelo A para generarlo.

from pathlib import Path as _Path
from datasets import Dataset, Features, Value
from datasets import Image as HFImage

if not COMMON_MANIFEST.exists():
    raise FileNotFoundError(
        f'No se encontro el manifiesto comun en: {COMMON_MANIFEST}\n'
        'Corre primero el notebook del Modelo A (seccion "Split comun '
        'reproducible MEL/NV: 70% train / 15% val / 15% test") para '
        'generar el manifiesto compartido. Modelo B necesita ese archivo '
        'para usar exactamente las mismas particiones train/validation/test.'
    )

manifest_df = pd.read_csv(COMMON_MANIFEST)
print('Manifiesto comun cargado:', COMMON_MANIFEST)
print(manifest_df['split'].value_counts())

required_cols = {'sample_id', 'split', 'binary_label', 'binary_label_name',
                  'source_split', 'source_index'}
missing_cols = required_cols - set(manifest_df.columns)
if missing_cols:
    raise ValueError(f'Al manifiesto le faltan columnas esperadas: {missing_cols}')


def build_split_from_manifest(manifest_df, split_name, raw_dataset, image_col):
    """Reconstruye un split de datasets.Dataset seleccionando EXACTAMENTE
    las mismas muestras (source_split, source_index) que uso el Modelo A,
    y les asigna la misma binary_label registrada en el manifiesto."""
    rows = manifest_df.loc[manifest_df['split'] == split_name].reset_index(drop=True)

    images, binary_labels, binary_label_names, sample_ids = [], [], [], []
    for _, row in rows.iterrows():
        example = raw_dataset[row['source_split']][int(row['source_index'])]
        images.append(example[image_col])
        binary_labels.append(int(row['binary_label']))
        binary_label_names.append(row['binary_label_name'])
        sample_ids.append(row['sample_id'])

    features = Features({
        image_col: HFImage(),
        'binary_label': Value('int64'),
        'binary_label_name': Value('string'),
        'sample_id': Value('string'),
    })

    return Dataset.from_dict({
        image_col: images,
        'binary_label': binary_labels,
        'binary_label_name': binary_label_names,
        'sample_id': sample_ids,
    }, features=features)


finetune_dataset = DatasetDict({
    split_name: build_split_from_manifest(manifest_df, split_name, dataset, IMAGE_COL)
    for split_name in ['train', 'validation', 'test']
})

# ----------------------------------------------------------------
# Verificaciones de integridad frente al manifiesto de Modelo A
# ----------------------------------------------------------------
for split_name in ['train', 'validation', 'test']:
    expected_n = int((manifest_df['split'] == split_name).sum())
    actual_n = len(finetune_dataset[split_name])
    assert actual_n == expected_n, (
        f'{split_name}: se esperaban {expected_n} muestras del manifiesto, '
        f'se obtuvieron {actual_n}.'
    )

train_ids = set(finetune_dataset['train']['sample_id'])
val_ids = set(finetune_dataset['validation']['sample_id'])
test_ids = set(finetune_dataset['test']['sample_id'])
assert train_ids.isdisjoint(val_ids), 'Fuga entre train y validation.'
assert train_ids.isdisjoint(test_ids), 'Fuga entre train y test.'
assert val_ids.isdisjoint(test_ids), 'Fuga entre validation y test.'

print('\n\u2713 Particiones de Modelo B identicas al manifiesto de Modelo A.')
for split_name in ['train', 'validation', 'test']:
    labels = finetune_dataset[split_name]['binary_label_name']
    print(f'{split_name}: n={len(finetune_dataset[split_name])}')
    print(pd.Series(labels).value_counts())


In [ ]:
# =========================================================
# 10. Imagenes usadas para el preentrenamiento SimCLR
# =========================================================
# SimCLR (autosupervisado) usa imagenes de las 8 clases de ISIC2019 sin
# etiquetas. PERO para que la evaluacion final en test sea honesta, estas
# imagenes deben excluir cualquier muestra que el manifiesto de Modelo A
# haya asignado a validation o test: de lo contrario el encoder "veria"
# (sin etiqueta, pero si los pixeles) las imagenes de test durante el
# preentrenamiento, lo cual es una fuga de datos.
USE_ALL_SPLITS_FOR_SIMCLR = False

excluded_pairs = set(
    zip(
        manifest_df.loc[manifest_df['split'].isin(['validation', 'test']), 'source_split'],
        manifest_df.loc[manifest_df['split'].isin(['validation', 'test']), 'source_index'].astype(int),
    )
)
print(f'Muestras excluidas de SimCLR por estar en val/test del manifiesto: {len(excluded_pairs)}')

def not_in_val_or_test(example, split_name):
    return (split_name, int(example['__orig_idx__'])) not in excluded_pairs

if USE_ALL_SPLITS_FOR_SIMCLR:
    from datasets import concatenate_datasets
    filtered_splits = [
        simclr_dataset[s].filter(lambda ex, s=s: not_in_val_or_test(ex, s))
        for s in simclr_dataset.keys()
    ]
    simclr_train = concatenate_datasets(filtered_splits)
else:
    simclr_train = simclr_dataset['train'].filter(lambda ex: not_in_val_or_test(ex, 'train'))

print('Imagenes en train crudo (8 clases, antes de excluir val/test):', len(simclr_dataset['train']))
print('Imagenes para SimCLR (tras excluir val/test del manifiesto):', len(simclr_train))


In [ ]:
# =========================================================
# 11. Transformaciones SimCLR y Fine-tuning
# =========================================================
# SimCLR original usa crop, color jitter, grayscale y blur. En dermatoscopía usamos jitter moderado.

class GaussianBlur:
    def __init__(self, kernel_size, sigma=(0.1, 2.0)):
        self.kernel_size = kernel_size
        self.sigma = sigma
    def __call__(self, img):
        sigma = random.uniform(self.sigma[0], self.sigma[1])
        return transforms.functional.gaussian_blur(img, self.kernel_size, [sigma, sigma])

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

simclr_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomApply([GaussianBlur(kernel_size=23)], p=0.5),
    transforms.ToTensor(),
    normalize,
])

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    normalize,
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize,
])


In [ ]:
# =========================================================
# 12. Datasets PyTorch
# =========================================================
def to_pil_image(value):
    """Convierte de forma robusta el valor devuelto por la columna de
    imagen del dataset a un PIL.Image, sin importar si viene:
    - ya decodificado como PIL.Image
    - como lista/tupla (algunas filas de este dataset traen la imagen
      envuelta en una lista de 1 elemento) -> se toma el primer elemento
    - como dict {'bytes': ..., 'path': ...} (formato Image(decode=False)
      de la libreria datasets)
    - como bytes/bytearray crudos
    - como ruta de archivo (str)
    """
    if isinstance(value, Image.Image):
        return value
    if isinstance(value, (list, tuple)):
        if len(value) == 0:
            raise ValueError('La columna de imagen vino como lista vacia.')
        return to_pil_image(value[0])
    if isinstance(value, dict):
        if value.get('bytes') is not None:
            return Image.open(io.BytesIO(value['bytes']))
        if value.get('path') is not None:
            return Image.open(value['path'])
        raise ValueError(f"No se pudo interpretar el dict de imagen: {list(value.keys())}")
    if isinstance(value, (bytes, bytearray)):
        return Image.open(io.BytesIO(value))
    # Se asume ruta de archivo u objeto tipo file-like abrible directamente
    return Image.open(value)

class SimCLRDataset(Dataset):
    def __init__(self, hf_dataset, image_col, transform):
        self.ds = hf_dataset
        self.image_col = image_col
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def _get_single_item(self, idx):
        """Helper to get and transform a single item."""
        img = to_pil_image(self.ds[idx][self.image_col])
        img = img.convert('RGB')
        x1 = self.transform(img)
        x2 = self.transform(img)
        return x1, x2

    def __getitem__(self, idx):
        # If a list of indices is passed (e.g., from DataLoader's __getitems__ or custom collation)
        if isinstance(idx, (list, tuple)):
            # Process each index in the list
            batch_x1 = []
            batch_x2 = []
            for i in idx:
                x1, x2 = self._get_single_item(i)
                batch_x1.append(x1)
                batch_x2.append(x2)
            # Stack them into a batch
            return torch.stack(batch_x1), torch.stack(batch_x2)
        else:
            # If a single index is passed
            return self._get_single_item(idx)

class ClassificationDataset(Dataset):
    def __init__(self, hf_dataset, image_col, label_col='binary_label', transform=None):
        self.ds = hf_dataset
        self.image_col = image_col
        self.label_col = label_col
        self.transform = transform
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        item = self.ds[idx]
        img = to_pil_image(item[self.image_col])
        img = img.convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = int(item[self.label_col])
        return img, label

simclr_loader = DataLoader(
    SimCLRDataset(simclr_train, IMAGE_COL, simclr_transform),
    batch_size=BATCH_SIZE_SIMCLR,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
)

train_loader = DataLoader(
    ClassificationDataset(finetune_dataset['train'], IMAGE_COL, transform=train_transform),
    batch_size=BATCH_SIZE_FINETUNE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
)

val_loader = DataLoader(
    ClassificationDataset(finetune_dataset['validation'], IMAGE_COL, transform=val_transform),
    batch_size=BATCH_SIZE_FINETUNE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
)

test_loader = DataLoader(
    ClassificationDataset(finetune_dataset['test'], IMAGE_COL, transform=val_transform),
    batch_size=BATCH_SIZE_FINETUNE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
)

print('Batches SimCLR:', len(simclr_loader))
print('Batches train:', len(train_loader), 'val:', len(val_loader), 'test:', len(test_loader))


In [ ]:
# =========================================================
# 13. Modelos: Encoder ResNet50, Projection Head, SimCLR Model
# =========================================================
class ResNet50Encoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        base = resnet50(weights=weights)
        self.features = nn.Sequential(*list(base.children())[:-1])  # hasta avgpool
        self.out_dim = base.fc.in_features
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return x

class ProjectionHead(nn.Module):
    # Projection head no lineal como SimCLR: Linear -> ReLU -> Linear
    def __init__(self, in_dim=2048, hidden_dim=2048, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim)
        )
    def forward(self, x):
        return self.net(x)

class SimCLRModel(nn.Module):
    def __init__(self, encoder, projection_head):
        super().__init__()
        self.encoder = encoder
        self.projection_head = projection_head
    def forward(self, x):
        h = self.encoder(x)
        z = self.projection_head(h)
        z = F.normalize(z, dim=1)
        return z

class LinearClassifier(nn.Module):
    def __init__(self, encoder, num_classes=2, dropout=0.0):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(encoder.out_dim, num_classes)
        )
    def forward(self, x):
        h = self.encoder(x)
        return self.classifier(h)


In [ ]:
# =========================================================
# 14. NT-Xent Loss de SimCLR
# =========================================================
def nt_xent_loss(z1, z2, temperature=0.5):
    # Implementación estándar de NT-Xent.
    # z1, z2: tensores normalizados [N, D]
    batch_size = z1.shape[0]
    z = torch.cat([z1, z2], dim=0)  # [2N, D]
    sim = torch.matmul(z, z.T) / temperature  # cosine porque z está normalizado

    # Enmascarar similitud consigo mismo
    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
    sim.masked_fill_(mask, torch.finfo(sim.dtype).min)

    # Positivos: i <-> i+N
    positives = torch.cat([
        torch.arange(batch_size, 2 * batch_size, device=z.device),
        torch.arange(0, batch_size, device=z.device)
    ], dim=0)

    loss = F.cross_entropy(sim, positives)
    return loss


In [ ]:
# =========================================================
# 15. Schedulers: Warmup + Cosine
# =========================================================
def build_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps = total_epochs * steps_per_epoch

    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step + 1) / float(max(1, warmup_steps))
        progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


In [ ]:
# =========================================================
# 16. Utilidades de checkpoints, CSV y evaluación
# =========================================================
def save_checkpoint(path, payload):
    torch.save(payload, path)


def load_checkpoint(path, map_location=DEVICE):
    return torch.load(path, map_location=map_location)


def extract_submodule_state_dict(full_state_dict, prefix):
    """Reconstruye el state_dict de un submodulo (p.ej. 'encoder' o
    'classifier') a partir del state_dict completo del modelo padre,
    sin necesidad de haber guardado una copia separada en el checkpoint.
    Antes guardabamos 'model_state_dict' junto con copias redundantes de
    cada submodulo ('encoder_state_dict', 'classifier_state_dict', etc.)
    en cada epoca; como esas copias ya estan contenidas dentro de
    'model_state_dict', ahora se derivan aqui bajo demanda en vez de
    duplicarse en disco (los checkpoints se escriben en Drive, asi que
    esto tambien ahorra tiempo de escritura por epoca)."""
    key_prefix = prefix + '.'
    return {
        k[len(key_prefix):]: v
        for k, v in full_state_dict.items()
        if k.startswith(key_prefix)
    }


def append_csv(path, row):
    path = Path(path)
    df = pd.DataFrame([row])
    if path.exists():
        df.to_csv(path, mode='a', header=False, index=False)
    else:
        df.to_csv(path, index=False)


def get_lr(optimizer):
    return optimizer.param_groups[0]['lr']

@torch.no_grad()
def evaluate_classifier(model, loader, criterion=None):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    total_loss = 0.0
    n = 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = torch.argmax(logits, dim=1)
        if criterion is not None:
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
        n += x.size(0)
        all_labels.extend(y.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)
    metrics = {
        'loss': total_loss / max(1, n) if criterion is not None else np.nan,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_mel': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall_mel': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1_mel': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }
    try:
        metrics['roc_auc'] = roc_auc_score(y_true, y_prob)
        metrics['pr_auc'] = average_precision_score(y_true, y_prob)
    except Exception:
        metrics['roc_auc'] = np.nan
        metrics['pr_auc'] = np.nan
    return metrics, y_true, y_pred, y_prob


In [ ]:
# =========================================================
# 17. Utilidades de bloqueo (lock) para entrenamiento entre varias cuentas
# =========================================================
import getpass, uuid, datetime

LOCK_STALE_MINUTES = 20
HEARTBEAT_INTERVAL_SEC = 180  # refresca el heartbeat cada 3 min durante el
                               # entrenamiento, para que el lock no expire
                               # aunque una epoca tarde mas de 20 min.
SESSION_ID = str(uuid.uuid4())[:8]

def _now_utc():
    return datetime.datetime.utcnow()

def acquire_lock(lock_path, owner_label):
    lock_path = Path(lock_path)
    if lock_path.exists():
        try:
            info = json.load(open(lock_path))
            last = datetime.datetime.fromisoformat(info['heartbeat'])
            age_min = (_now_utc() - last).total_seconds() / 60
            if age_min < LOCK_STALE_MINUTES and info.get('session_id') != SESSION_ID:
                raise RuntimeError(
                    f"Otra sesión (cuenta/persona: '{info.get('owner')}') parece estar "
                    f"entrenando ahora mismo (último heartbeat hace {age_min:.1f} min, "
                    f"límite {LOCK_STALE_MINUTES} min).\n"
                    "Si estás SEGURO de que esa sesión ya terminó o se colgó, borra "
                    f"manualmente este archivo y vuelve a correr la celda:\n{lock_path}"
                )
        except (json.JSONDecodeError, KeyError, ValueError):
            pass
    write_heartbeat(lock_path, owner_label)
    print(f"Lock adquirido por '{owner_label}' (sesión {SESSION_ID}) en {lock_path}")

def write_heartbeat(lock_path, owner_label):
    json.dump(
        {'owner': owner_label, 'session_id': SESSION_ID, 'heartbeat': _now_utc().isoformat()},
        open(lock_path, 'w')
    )

def release_lock(lock_path):
    try:
        Path(lock_path).unlink()
        print(f'Lock liberado: {lock_path}')
    except FileNotFoundError:
        pass

try:
    RUNNER_LABEL = input(
        'Nombre o cuenta de quien corre este entrenamiento ahora (para el registro): '
    ).strip()
except Exception:
    RUNNER_LABEL = ''
if not RUNNER_LABEL:
    RUNNER_LABEL = getpass.getuser()


In [ ]:
# =========================================================
# 18. Preentrenamiento SimCLR con checkpoints y reanudación automática
# =========================================================
encoder = ResNet50Encoder(pretrained=True).to(DEVICE)
projection_head = ProjectionHead(encoder.out_dim, HIDDEN_DIM, PROJECTION_DIM).to(DEVICE)
simclr_model = SimCLRModel(encoder, projection_head).to(DEVICE)

optimizer_simclr = optim.AdamW(simclr_model.parameters(), lr=SIMCLR_LR, weight_decay=SIMCLR_WEIGHT_DECAY)
steps_per_epoch_simclr = math.ceil(len(simclr_loader) / GRAD_ACCUM_STEPS_SIMCLR)
scheduler_simclr = build_warmup_cosine_scheduler(
    optimizer_simclr, WARMUP_EPOCHS_SIMCLR, SIMCLR_EPOCHS, steps_per_epoch_simclr
)
scaler_simclr = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE.type == 'cuda'))

simclr_last_path = SIMCLR_CKPT_DIR / 'simclr_last.pth'
simclr_best_path = SIMCLR_CKPT_DIR / 'simclr_best_loss.pth'
simclr_history_path = HISTORY_DIR / 'history_simclr.csv'
simclr_lock_path = SIMCLR_CKPT_DIR / 'TRAINING.lock'

# Evita que dos cuentas entrenen SimCLR al mismo tiempo sobre la misma carpeta.
acquire_lock(simclr_lock_path, RUNNER_LABEL)

start_epoch = 1
best_simclr_loss = float('inf')
global_step = 0
epochs_without_improvement_simclr = 0

if AUTO_RESUME_SIMCLR and simclr_last_path.exists():
    ckpt = load_checkpoint(simclr_last_path)
    simclr_model.load_state_dict(ckpt['model_state_dict'])
    optimizer_simclr.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler_simclr.load_state_dict(ckpt['scheduler_state_dict'])
    scaler_simclr.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_simclr_loss = ckpt.get('best_loss', best_simclr_loss)
    global_step = ckpt.get('global_step', 0)
    # Fix: antes esto se reseteaba a 0 sin leer el checkpoint, perdiendo el
    # progreso del early stopping cada vez que se reanudaba.
    epochs_without_improvement_simclr = ckpt.get('epochs_without_improvement_simclr', 0)
    print(f'Reanudando SimCLR desde época {start_epoch}. Mejor loss: {best_simclr_loss:.4f}. '
          f'Sin mejora: {epochs_without_improvement_simclr}')
else:
    print('Iniciando SimCLR desde cero')

writer_simclr = SummaryWriter(log_dir=str(LOG_DIR / 'simclr'))

import torch.cuda.amp as amp

SIMCLR_EARLY_STOP_PATIENCE = 20
last_heartbeat_time = time.time()

# Ejecutar entrenamiento SimCLR
for epoch in range(start_epoch, SIMCLR_EPOCHS + 1):
    simclr_model.train()
    running_loss = 0.0
    optimizer_simclr.zero_grad(set_to_none=True)
    start_time = time.time()

    for step, (x1, x2) in enumerate(simclr_loader, start=1):
        x1 = x1.to(DEVICE, non_blocking=True)
        x2 = x2.to(DEVICE, non_blocking=True)

        with amp.autocast(enabled=(USE_AMP and DEVICE.type == 'cuda')):
            z1 = simclr_model(x1)
            z2 = simclr_model(x2)
            loss = nt_xent_loss(z1, z2, temperature=SIMCLR_TEMPERATURE)
            loss = loss / GRAD_ACCUM_STEPS_SIMCLR

        scaler_simclr.scale(loss).backward()

        if step % GRAD_ACCUM_STEPS_SIMCLR == 0:
            scaler_simclr.step(optimizer_simclr)
            scaler_simclr.update()
            optimizer_simclr.zero_grad(set_to_none=True)
            scheduler_simclr.step()
            global_step += 1

        running_loss += loss.item() * GRAD_ACCUM_STEPS_SIMCLR

        # Refresca el heartbeat dentro de la epoca (no solo al final),
        # para que el lock no parezca abandonado si una epoca es larga.
        if time.time() - last_heartbeat_time > HEARTBEAT_INTERVAL_SEC:
            write_heartbeat(simclr_lock_path, RUNNER_LABEL)
            last_heartbeat_time = time.time()

    epoch_loss = running_loss / len(simclr_loader)
    lr = get_lr(optimizer_simclr)
    elapsed = time.time() - start_time

    if epoch_loss < best_simclr_loss:
        best_simclr_loss = epoch_loss
        epochs_without_improvement_simclr = 0
        is_best = True
    else:
        epochs_without_improvement_simclr += 1
        is_best = False

    print(f'[SimCLR] Epoch {epoch:03d}/{SIMCLR_EPOCHS} | loss={epoch_loss:.4f} | lr={lr:.2e} | time={elapsed/60:.1f} min')
    writer_simclr.add_scalar('loss/train', epoch_loss, epoch)
    writer_simclr.add_scalar('lr', lr, epoch)

    row = {'epoch': epoch, 'loss': epoch_loss, 'lr': lr, 'time_sec': elapsed, 'runner': RUNNER_LABEL}
    append_csv(simclr_history_path, row)

    write_heartbeat(simclr_lock_path, RUNNER_LABEL)
    last_heartbeat_time = time.time()

    payload = {
        'epoch': epoch,
        # Antes: tambien se guardaban 'encoder_state_dict' y
        # 'projection_head_state_dict' aqui, duplicando pesos que ya
        # estan dentro de 'model_state_dict'. Se eliminan esas copias;
        # se reconstruyen con extract_submodule_state_dict() cuando
        # hacen falta (ver seccion 19).
        'model_state_dict': simclr_model.state_dict(),
        'optimizer_state_dict': optimizer_simclr.state_dict(),
        'scheduler_state_dict': scheduler_simclr.state_dict(),
        'scaler_state_dict': scaler_simclr.state_dict(),
        'best_loss': best_simclr_loss,
        'global_step': global_step,
        'epochs_without_improvement_simclr': epochs_without_improvement_simclr,
        'config': config,
    }
    save_checkpoint(simclr_last_path, payload)

    if is_best:
        payload['best_loss'] = best_simclr_loss
        save_checkpoint(simclr_best_path, payload)
        print('  ✓ Nuevo mejor checkpoint SimCLR guardado')
    else:
        print(f'  Sin mejora: {epochs_without_improvement_simclr}/{SIMCLR_EARLY_STOP_PATIENCE}')

    if epochs_without_improvement_simclr >= SIMCLR_EARLY_STOP_PATIENCE:
        print('Early stopping activado para SimCLR.')
        break

writer_simclr.close()
release_lock(simclr_lock_path)
print('Preentrenamiento SimCLR terminado.')


In [ ]:
# =========================================================
# 19. Cargar mejor encoder SimCLR para fine-tuning
# =========================================================
# Si existe simclr_best_loss.pth, se utiliza para inicializar el fine-tuning.
encoder_ft = ResNet50Encoder(pretrained=False).to(DEVICE)

if simclr_best_path.exists():
    ckpt = load_checkpoint(simclr_best_path)
    # El checkpoint ya no guarda 'encoder_state_dict' por separado (ver
    # nota en la seccion 18); se extrae del 'model_state_dict' completo.
    encoder_ft.load_state_dict(extract_submodule_state_dict(ckpt['model_state_dict'], 'encoder'))
    print('Encoder cargado desde mejor checkpoint SimCLR:', simclr_best_path)
else:
    print('No se encontró checkpoint SimCLR. Usando encoder aleatorio/predefinido.')


In [ ]:
# =========================================================
# 20. Fine-tuning supervisado MEL vs NV
# =========================================================
classifier_model = LinearClassifier(encoder_ft, num_classes=2, dropout=0.2).to(DEVICE)

# Pesos de clase para compensar desbalance
if USE_CLASS_WEIGHTS:
    train_labels = np.array(finetune_dataset['train']['binary_label'])
    counts = np.bincount(train_labels, minlength=2)
    weights = counts.sum() / (2.0 * counts)
    class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    print('Class weights [NV, MEL]:', class_weights)
    criterion_ft = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion_ft = nn.CrossEntropyLoss()

# LR diferencial: backbone bajo, FC alto
optimizer_ft = optim.AdamW([
    {'params': classifier_model.encoder.parameters(), 'lr': FINETUNE_LR_BACKBONE},
    {'params': classifier_model.classifier.parameters(), 'lr': FINETUNE_LR_FC},
], weight_decay=FINETUNE_WEIGHT_DECAY)

steps_per_epoch_ft = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS_FINETUNE)
scheduler_ft = build_warmup_cosine_scheduler(
    optimizer_ft, WARMUP_EPOCHS_FINETUNE, FINETUNE_EPOCHS, steps_per_epoch_ft
)
scaler_ft = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE.type == 'cuda'))

ft_last_path = FINETUNE_CKPT_DIR / 'finetune_last.pth'
ft_best_path = FINETUNE_CKPT_DIR / 'finetune_best_f1.pth'
ft_history_path = HISTORY_DIR / 'history_finetune.csv'
ft_lock_path = FINETUNE_CKPT_DIR / 'TRAINING.lock'

# Evita que dos cuentas entrenen fine-tuning al mismo tiempo sobre la misma carpeta.
acquire_lock(ft_lock_path, RUNNER_LABEL)

start_epoch_ft = 1
best_val_f1 = -1.0
epochs_without_improvement = 0
global_step_ft = 0

if AUTO_RESUME_FINETUNE and ft_last_path.exists():
    ckpt = load_checkpoint(ft_last_path)
    classifier_model.load_state_dict(ckpt['model_state_dict'])
    optimizer_ft.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler_ft.load_state_dict(ckpt['scheduler_state_dict'])
    scaler_ft.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch_ft = ckpt['epoch'] + 1
    best_val_f1 = ckpt.get('best_val_f1', best_val_f1)
    epochs_without_improvement = ckpt.get('epochs_without_improvement', 0)
    global_step_ft = ckpt.get('global_step', 0)
    print(f'Reanudando fine-tuning desde época {start_epoch_ft}. Mejor F1 val: {best_val_f1:.4f}')
else:
    print('Iniciando fine-tuning desde cero')

writer_ft = SummaryWriter(log_dir=str(LOG_DIR / 'finetuning'))

# Ejecutar fine-tuning
last_heartbeat_time_ft = time.time()

for epoch in range(start_epoch_ft, FINETUNE_EPOCHS + 1):
    classifier_model.train()
    running_loss = 0.0
    n_train = 0
    optimizer_ft.zero_grad(set_to_none=True)
    start_time = time.time()

    for step, (x, y) in enumerate(train_loader, start=1):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(USE_AMP and DEVICE.type == 'cuda')):
            logits = classifier_model(x)
            loss = criterion_ft(logits, y)
            loss = loss / GRAD_ACCUM_STEPS_FINETUNE

        scaler_ft.scale(loss).backward()

        if step % GRAD_ACCUM_STEPS_FINETUNE == 0:
            scaler_ft.step(optimizer_ft)
            scaler_ft.update()
            optimizer_ft.zero_grad(set_to_none=True)
            scheduler_ft.step()
            global_step_ft += 1

        running_loss += loss.item() * GRAD_ACCUM_STEPS_FINETUNE * x.size(0)
        n_train += x.size(0)

        # Igual que en SimCLR: refresca el heartbeat dentro de la epoca.
        if time.time() - last_heartbeat_time_ft > HEARTBEAT_INTERVAL_SEC:
            write_heartbeat(ft_lock_path, RUNNER_LABEL)
            last_heartbeat_time_ft = time.time()

    train_loss = running_loss / max(1, n_train)
    val_metrics, _, _, _ = evaluate_classifier(classifier_model, val_loader, criterion_ft)
    elapsed = time.time() - start_time

    print(f"[FT] Epoch {epoch:03d}/{FINETUNE_EPOCHS} | train_loss={train_loss:.4f} | "
          f"val_loss={val_metrics['loss']:.4f} | val_acc={val_metrics['accuracy']:.4f} | "
          f"val_f1={val_metrics['f1_mel']:.4f} | time={elapsed/60:.1f} min")

    writer_ft.add_scalar('loss/train', train_loss, epoch)
    writer_ft.add_scalar('loss/val', val_metrics['loss'], epoch)
    writer_ft.add_scalar('metrics/val_accuracy', val_metrics['accuracy'], epoch)
    writer_ft.add_scalar('metrics/val_f1_mel', val_metrics['f1_mel'], epoch)
    writer_ft.add_scalar('metrics/val_precision_mel', val_metrics['precision_mel'], epoch)
    writer_ft.add_scalar('metrics/val_recall_mel', val_metrics['recall_mel'], epoch)
    writer_ft.add_scalar('lr/backbone', optimizer_ft.param_groups[0]['lr'], epoch)
    writer_ft.add_scalar('lr/fc', optimizer_ft.param_groups[1]['lr'], epoch)

    write_heartbeat(ft_lock_path, RUNNER_LABEL)
    last_heartbeat_time_ft = time.time()

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        **{f'val_{k}': v for k, v in val_metrics.items()},
        'lr_backbone': optimizer_ft.param_groups[0]['lr'],
        'lr_fc': optimizer_ft.param_groups[1]['lr'],
        'time_sec': elapsed,
    }
    append_csv(ft_history_path, row)

    payload = {
        'epoch': epoch,
        # Antes: tambien se guardaban 'encoder_state_dict' y
        # 'classifier_state_dict' aqui, duplicando pesos que ya estan
        # dentro de 'model_state_dict'. Se eliminan esas copias; se
        # reconstruyen con extract_submodule_state_dict() si hacen falta.
        'model_state_dict': classifier_model.state_dict(),
        'optimizer_state_dict': optimizer_ft.state_dict(),
        'scheduler_state_dict': scheduler_ft.state_dict(),
        'scaler_state_dict': scaler_ft.state_dict(),
        'best_val_f1': best_val_f1,
        'epochs_without_improvement': epochs_without_improvement,
        'global_step': global_step_ft,
        'config': config,
    }
    save_checkpoint(ft_last_path, payload)

    if val_metrics['f1_mel'] > best_val_f1:
        best_val_f1 = val_metrics['f1_mel']
        epochs_without_improvement = 0
        payload['best_val_f1'] = best_val_f1
        payload['epochs_without_improvement'] = epochs_without_improvement
        save_checkpoint(ft_best_path, payload)
        print('  ✓ Nuevo mejor checkpoint fine-tuning guardado')
    else:
        epochs_without_improvement += 1
        print(f'  Sin mejora: {epochs_without_improvement}/{EARLY_STOP_PATIENCE}')

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print('Early stopping activado.')
        break

writer_ft.close()
release_lock(ft_lock_path)
print('Fine-tuning terminado. Mejor val F1:', best_val_f1)


In [ ]:
# =========================================================
# 21. Evaluación final en TEST
# =========================================================
# Cargar mejor modelo por F1 de validación
if ft_best_path.exists():
    ckpt = load_checkpoint(ft_best_path)
    classifier_model.load_state_dict(ckpt['model_state_dict'])
    print('Cargado mejor modelo fine-tuning:', ft_best_path)

# Evaluar
test_metrics, y_true, y_pred, y_prob = evaluate_classifier(classifier_model, test_loader, criterion_ft)
print('Resultados en TEST')
for k, v in test_metrics.items():
    print(f'{k}: {v:.4f}')

print('\nReporte de clasificación')
print(classification_report(y_true, y_pred, target_names=['NV', 'MEL'], digits=4))

# Guardar métricas test
test_metrics_path = HISTORY_DIR / 'test_metrics.csv'
pd.DataFrame([test_metrics]).to_csv(test_metrics_path, index=False)
print('Métricas test guardadas en:', test_metrics_path)


In [ ]:
# =========================================================
# 22. Matriz de confusión
# =========================================================
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm)
ax.set_title('Matriz de confusión - Modelo B SimCLR + Fine-tuning')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['NV', 'MEL']); ax.set_yticklabels(['NV', 'MEL'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center')
fig.colorbar(im, ax=ax)
plt.tight_layout()
fig_path = FIGURES_DIR / 'confusion_matrix_test.png'
plt.savefig(fig_path, dpi=200)
plt.show()
print('Figura guardada en:', fig_path)


In [ ]:
# =========================================================
# 23. Exportar encoder listo para TCAV
# =========================================================
# Este archivo contiene únicamente el encoder ResNet50 entrenado con SimCLR + Fine-tuning.
# Para TCAV, puedes cargarlo y extraer activaciones de layer1, layer2, layer3, layer4 o avgpool.

encoder_export_path = EXPORT_DIR / 'encoder_simclr_finetuned_ready_for_tcav.pth'
full_model_export_path = EXPORT_DIR / 'model_b_simclr_finetuned_full.pth'

export_payload = {
    'encoder_state_dict': classifier_model.encoder.state_dict(),
    'encoder_out_dim': classifier_model.encoder.out_dim,
    'image_size': IMG_SIZE,
    'normalization_mean': [0.485, 0.456, 0.406],
    'normalization_std': [0.229, 0.224, 0.225],
    'labels': {'NV': 0, 'MEL': 1},
    'source': 'SimCLR pretraining on 8 ISIC2019 classes + supervised fine-tuning on MEL vs NV',
    'config': config,
}
torch.save(export_payload, encoder_export_path)

torch.save({
    'model_state_dict': classifier_model.state_dict(),
    'labels': {'NV': 0, 'MEL': 1},
    'config': config,
}, full_model_export_path)

print('Encoder listo para TCAV guardado en:', encoder_export_path)
print('Modelo completo guardado en:', full_model_export_path)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

history_file = HISTORY_DIR / 'history_simclr.csv'

if history_file.exists():
    df = pd.read_csv(history_file)

    # Plotting SimCLR training loss
    plt.figure(figsize=(10, 6))
    plt.plot(df['epoch'], df['loss'], label='SimCLR Training Loss')
    plt.title('SimCLR Training Loss over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    figures_dir = FIGURES_DIR
    figures_dir.mkdir(parents=True, exist_ok=True)
    plot_path = figures_dir / 'simclr_training_loss.png'
    plt.savefig(plot_path)
    plt.show()
    print(f"Plot saved to: {plot_path}")

    print("Note: The 'history_simclr.csv' file only contains the training loss for the SimCLR pretraining. Validation loss, train accuracy, and validation accuracy are typically recorded during the supervised fine-tuning phase, which would be in a different history file (e.g., 'history_finetune.csv').")

else:
    print(f"Error: History file not found at {history_file}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ft_history_path = HISTORY_DIR / 'history_finetune.csv' # Using the previously defined HISTORY_DIR

if ft_history_path.exists():
    df_ft = pd.read_csv(ft_history_path)

    plt.figure(figsize=(15, 6))

    # Plotting Training Loss and Validation Loss
    plt.subplot(1, 2, 1)
    plt.plot(df_ft['epoch'], df_ft['train_loss'], label='Fine-tune Training Loss')
    plt.plot(df_ft['epoch'], df_ft['val_loss'], label='Fine-tune Validation Loss')
    plt.title('Fine-tuning Loss over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Plotting Validation Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(df_ft['epoch'], df_ft['val_accuracy'], label='Fine-tune Validation Accuracy', color='green')
    plt.title('Fine-tuning Validation Accuracy over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    figures_dir = FIGURES_DIR # Using the previously defined FIGURES_DIR
    figures_dir.mkdir(parents=True, exist_ok=True)
    plot_path = figures_dir / 'finetune_loss_accuracy_plots.png'
    plt.savefig(plot_path)
    plt.show()
    print(f"Fine-tuning plots saved to: {plot_path}")

    print("Note: Training accuracy for the fine-tuning phase is not available in 'history_finetune.csv'. Only training loss, validation loss, and validation accuracy were logged.")

elif not ft_history_path.exists():
    print(f"Error: Fine-tuning history file not found at {ft_history_path}. Please ensure the fine-tuning process has been completed and the history file exists.")


## Notas para el artículo

Para una comparación justa contra el Modelo A:

1. Usa el mismo split MEL vs NV para fine-tuning y evaluación.
2. Reporta Accuracy, Precision, Recall, F1, ROC-AUC y PR-AUC.
3. Si el Modelo B queda muy por debajo del Modelo A, aumenta `SIMCLR_EPOCHS`, `BATCH_SIZE_SIMCLR` o usa `GRAD_ACCUM_STEPS_SIMCLR`.
4. No pases a TCAV hasta que ambos modelos tengan rendimiento comparable.
5. Para TCAV, carga `encoder_simclr_finetuned_ready_for_tcav.pth` y extrae activaciones de la misma capa que uses en el Modelo A.